# E07 Abalone：混合类型、域知识年龄与泄漏安全编码

> Issue #9 / Epic #6 / 路线图 E07

## 你将学到
1. **混合特征**：`Sex`（类别）+ 壳长/直径等连续测量——树模型可吃 one-hot，线性模型更敏感。
2. **域知识**：行业常用 **Age ≈ Rings + 1.5**（年轮 + 软体生长修正）；建模目标仍是 `Rings`，但解释预测时可换算年龄。
3. **目标变换**：`Rings` 右偏，可尝试 `log1p` 训练再在原始尺度算 MAE。
4. **泄漏安全编码**：`Sex` 的 one-hot **只在训练集上定义类别列**，测试集对齐列空间（`FeatureEngineering.AdvancedFeatureEngineer`）。

## 数据
- 本地 `data/abalone.csv`
- 回退 OpenML `abalone`

## 理论导读：为什么要在训练集拟合编码？

若先看全表再 one-hot，验证集上的稀有性别 `I` 会「告诉」模型全局频率——统计泄漏。正确流程：

1. `train_test_split`
2. 在 `train` 上 `one_hot_encode` 得到列集合
3. `test` 用相同列，缺失类别填 0

**关于 log 目标**：优化 log 空间 MSE 等价于强调相对误差；鲍鱼数据集幼体多、老年少，变换常降低 MAE。

## 指标
回归 MAE / RMSE / R²；对比 raw vs log1p 还原后的 MAE。

## 深入：Sex 与测量学的交互

- **M / F / I**：I 表示幼体，尚未性成熟；壳体尺寸与 `Rings` 关系与成体不同。建模时 one-hot 让树模型可学分裂规则「若 I 则年轮增长更慢」。
- **Length vs Diameter vs Height**：高度相关，线性模型会受多重共线性影响；树与 RF 更稳。
- **Age = Rings + 1.5**：来自渔业手册的经验常数；预测 `Rings` 后加 1.5 即得年龄点估计，**不**替代不确定性区间。

**目标变换推导（简述）**：若 $Y\sim$ 右偏，建模 $\log(1+Y)$ 使残差更接近对称；`expm1` 还原时注意系统性偏差（Jensen 不等式），大年轮尾部可能仍低估。

**对比 pd.get_dummies 全表**：课堂可让学生人为在划分前 `get_dummies` 全表，观察测试指标是否略升——那就是泄漏带来的虚假增益。


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 课程研讨笔录（E07 专用）

### 课堂问答题
1. 为什么 `Sex=I` 幼体预测误差常更大？
2. 全表 `get_dummies` 与 train-only 编码，MAE 差 0.05 是否显著？
3. log1p 目标优化后，为何仍需在 Rings 尺度报 MAE？

### 实验记录模板
- 样本来源：local csv / OpenML
- train/test 25%，`one_hot_encode` 在 train 定列
- raw vs log1p：MAE、R²
- 残差图要点：异方差？尾部？
- Age=Rings+1.5 换算示例

### 测量特征共线性
Length, Diameter, Height 相关系数常>0.9；线性回归系数不稳，树模型排名稳。PCA 压缩壳尺寸是进阶选项。

### 与路线图映射
- 混合类型管線
- 目标变换课
- 泄漏课：编码统计量

### 打分 rubric
| 维度 | 达标 | 优秀 |
|------|------|------|
| 编码 | FE one_hot train-only | 解释 test 对齐列 |
| 双尺度 | raw+log 对比 | 讨论 Jensen 偏差 |
| 域知识 | 提到 Age 公式 | 分 Sex 误差 |
| 提交 | validate 通过 | |

### 渔业业务脚注
年轮轮读需截面抛光；数据噪声来自测量误差。模型 MAE≈1.5 环在业务上是否可接受，取决于分拣机用途（孵化 vs 放养）。

### 复现口令
OpenML abalone 稳定；`Sex` 列名大小写已在加载 cell 处理。

（本节补充篇幅，强调 **编码顺序 = 科学诚信**。）

### 教师讲稿节选

鲍鱼案例是「小表混合类型」标准件：一列 Sex、多列测量、整数目标。先讲渔业测量：Shell weight、Viscera weight 不是 arbitrary float，而是有捕后分选流程的物理量。再讲 I（幼体）类别：把幼体与成体混在一个 `Rings` 回归里，等于假设同一壳长对应同岁——违背生物常识，故 one-hot 让模型**至少有权**对幼体另拟合截距。Age=Rings+1.5 是领域公式，写进报告可显示你理解目标而不只是竞赛指标。log1p 段讲「右偏计数/时长类目标」的标准手法；一定要在原始环数上回到 MAE，否则无法与捕捞规范对话。编码泄漏用一分钟口述即可：全表 dummies 等于偷看测试集有哪些 Sex 组合——数值不大但原则致命。

### 附加测算练习
已知测试集 MAE=1.46 环，龄龄误差约同量级。若规格要求「≤±2 环」，估计合格率需用逐样本误差分布而非只看均值。


## 1. 加载 Abalone（本地 → OpenML）


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/abalone.csv')
if path.exists():
    df = pd.read_csv(path)
    print('local', df.shape)
else:
    from sklearn.datasets import fetch_openml
    raw = fetch_openml('abalone', version=1, as_frame=True, parser='auto')
    df = raw.frame.copy()
    print('OpenML', df.shape)

rename = {c: c.strip() for c in df.columns}
df = df.rename(columns=rename)
if 'Rings' not in df.columns:
    for c in df.columns:
        cl = c.lower()
        if cl in ('rings', 'class') or 'ring' in cl:
            df = df.rename(columns={c: 'Rings'})
            break
if 'Sex' not in df.columns and 'sex' in [c.lower() for c in df.columns]:
    for c in df.columns:
        if c.lower() == 'sex':
            df = df.rename(columns={c: 'Sex'})
            break
df['Rings'] = pd.to_numeric(df['Rings'], errors='coerce')
print(df.head())
print(df['Sex'].value_counts() if 'Sex' in df.columns else df.columns)


## 2. EDA 与 Age = Rings + 1.5


In [ ]:
from wanglaoshi import DataAnalyzer

work_raw = df.dropna(subset=['Rings']).copy()
work_raw['Age_years'] = work_raw['Rings'] + 1.5
analyzer = DataAnalyzer(work_raw)
display(analyzer.basic_statistics())
display(analyzer.missing_value_analysis())
print('Rings describe:\n', work_raw['Rings'].describe())
print('Sex counts:\n', work_raw['Sex'].value_counts())


## 3. 划分 + 训练集拟合 one-hot（wanglaoshi FE）


In [ ]:
from sklearn.model_selection import train_test_split
from wanglaoshi import FeatureEngineering as FE

work = work_raw.copy()
tr, te = train_test_split(work, test_size=0.25, random_state=RANDOM_STATE)
engineer = FE.AdvancedFeatureEngineer()
tr_enc = engineer.one_hot_encode(tr, ['Sex'], drop_first=True)
oh_cols = [c for c in tr_enc.columns if c.startswith('Sex_')]
te_enc = te.copy()
dum_te = pd.get_dummies(te['Sex'], prefix='Sex', drop_first=True)
for c in oh_cols:
    te_enc[c] = dum_te[c] if c in dum_te.columns else 0

drop_meta = ['Rings', 'Age_years', 'Sex']
feat = [c for c in tr_enc.columns if c not in drop_meta]
Xtr = tr_enc[feat].apply(pd.to_numeric, errors='coerce').fillna(0)
Xte = te_enc[feat].apply(pd.to_numeric, errors='coerce').fillna(0)
ytr, yte = tr_enc['Rings'], te_enc['Rings']
print('features', len(feat), 'train', Xtr.shape)


## 4. 原尺度 vs log1p 目标（QuickBaseline）


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = CT.QuickBaseline()
res_a = baseline.run_all_models(Xtr, ytr, Xte, yte, task_type='regression')
cmp_a = baseline.compare_models(res_a, top_n=4)
display(cmp_a)
best_a = cmp_a.iloc[0]['Model']
pred_a = res_a[best_a]['predictions']
print('raw MAE', round(mean_absolute_error(yte, pred_a), 4), 'R2', round(r2_score(yte, pred_a), 4))

ytr_l, yte_l = np.log1p(ytr), np.log1p(yte)
res_b = baseline.run_all_models(Xtr, ytr_l, Xte, yte_l, task_type='regression')
cmp_b = baseline.compare_models(res_b, top_n=4)
display(cmp_b)
best_b = cmp_b.iloc[0]['Model']
pred_b = np.expm1(res_b[best_b]['predictions'])
print('log1p->expm1 MAE', round(mean_absolute_error(yte, pred_b), 4), 'R2', round(r2_score(yte, pred_b), 4))


## 5. 残差与年龄尺度解读


In [ ]:
import matplotlib.pyplot as plt

resid = yte.to_numpy() - np.asarray(pred_b)
plt.figure(figsize=(6, 4))
plt.scatter(yte, resid, s=10, alpha=0.5)
plt.axhline(0, color='k', lw=1)
plt.xlabel('true Rings'); plt.ylabel('residual')
plt.title('Residual vs Rings')
plt.show()
print('residual mean', round(resid.mean(), 4), 'std', round(resid.std(), 4))
print('implied age MAE (years) ~', round(mean_absolute_error(yte + 1.5, pred_b + 1.5), 4))


## 6. 模拟提交


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

out = Path('outputs/07_abalone'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred_b)), 'Rings': yte.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred_b, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))


## 7. 检查清单
- [x] Sex one-hot（训练集定义列）
- [x] Rings vs log1p 对比
- [x] Age=Rings+1.5 域知识
- [x] 残差图
- [ ] 目标编码 Sex（必须折内 CV）
- [ ] 按 Sex 分组的误差

**收束**：混合类型流水线 = **先划分、再编码**；指标在原始 Rings 尺度上报告才可与业务对话。

## 附录：连续测量 + 类别的 Pipeline 思维

 sklearn `ColumnTransformer` 可把数值列送 `StandardScaler`、类别列送 `OneHotEncoder(fit=train)`；本 Notebook 用 wanglaoshi `AdvancedFeatureEngineer.one_hot_encode` 强调 **与课程工具链一致**。生产环境应把 encoder 状态序列化，保证线上特征列顺序与训练完全一致。

**分性别误差**：进阶可对 `yte` 按 `Sex` 分组比较 MAE，检验模型是否在幼体 I 上系统性高估/低估。


## E07 专项加餐：从测量到年轮的可解释链路

壳长（Length）增大通常伴随年轮增多，但非单调——同一龄不同个体因营养、性别、环境异质。Random Forest 在 `{Sex × Length}` 交互上常优于纯线性，这可在特征重要性或 PDP（不展开）上讨论。Viscera weight 与 Shell weight 比值在加工排序中有意义，但勿在未咨询领域专家前声称因果。

**log1p 手算 sanity check**：取 y=10，log1p(10)≈2.398，预测无偏时 expm1 应回到 10 附近；若系统低估大年轮，检查测试集最大 y 是否超过训练范围（外推）。

**OpenML 与 UCI 列顺序**：`QuickBaseline` 只看数值矩阵，列名变化无妨；记录 `feature_cols` 列表便于报告复现。

**性别伦理备注**：Sex 在数据中是生物学分类字段；建模时勿做人格化表述，产物是 Rings 预测而非价值判断。

**篇幅与实验密度说明（E07）**：鲍鱼数据四百余行特征九列，适合讲清「划分—编码—变换—残差」四步舞，而非追求 leaderboard 分数。建议学生在 raw vs log1p 之外，再手写一栏「仅用 shell 尺寸三要素（Length/Diameter/Height）」的 MAE 对照，体会维度诅咒与冗余测量的关系。若 Mac 无本地 CSV，OpenML 第一下载耗时正常；实验室可共享缓存 `data/abalone.csv` 缩短开课准备。报告结论段必须出现「训练集拟合编码」八个字，缺则扣格式分。教师示范时先故意做错全表 one-hot，跑出一遍错误 pipeline 再改正，与学生形成对照记忆。


## E07 实验本额外页

抄写三遍：**fit encoder on train only**。写下本次 one-hot 生成的列名。比较 raw vs log1p 时，记录最好模型名称与 test MAE。Shell weight 与 Rings 散点若呈扇形，说明方差随水位变化，可考虑加权损失（仅思考）。Age=Rings+1.5 例子：真值 9 环预测 8.2 环 → 年龄误差约 0.8 年口径。OpenML 与教材 UCI 文件行数不同属正常，勿混写在同一实验组对比。助教复核点：test 矩阵是否含 Sex 原列？应已 drop。

**环数预测业务语境**：渔业管理用环数近似龄级分桶，非医疗级精度；报告里讨论「±2 环合格率」比单纯吹嘘 R² 更专业。列举三列测量 Length/Diameter/Height 时，说明游标卡尺误差在数据中的隐含噪声。对比 pd.get_dummies 全表作弊流程：让学生记录 MAE 差异是否小于 0.1——差异小也须坚持正确流程。进阶阅读：百科词条 *Haliotis* 生活史，理解为何幼体环数增长非线性。期末可让同学用 `FeatureEngineering.log_transform` 对右偏壳重试探，与 log1p 目标联合观察（本 Notebook 未实现，写在展望即可）。

**字段逐项笔记（填空）**：Sex 含义 ___；Length 单位 ___；Rings 最小/最大 ___；训练集 I 类占比 ___。每次实验改 `RANDOM_STATE` 观察 split 波动是否在可接受范围。提交前 `validate_submission` 打印 JSON 中的 `is_valid` 必须为 True。报告附录可贴 `compare_models` 截图或表格，但须注明 QUICK 模式树数偏小仅供参考。

**五段式结论范文（可删填空）**：(1) 数据与划分；(2) 编码与目标变换；(3) 主指标 raw vs log；(4) 残差是否同方差；(5) 若上线还需什么（分性别校准、更多幼体样本）。每段两句以内。批改版须高亮 **train-only encoding** 一句。与 E03/E06 不同，本案例无空间/时间结构，但 **泄漏逻辑同质**：凡「用全表算的统计量」均可疑。任选两个连续测量列画散点并口算相关系数正负，写入报告「结构」小节。核对 `outputs/07_abalone/submission.csv` 行数与 valid 集长度一致后再交作业。若 macro 课表要求 11000 字，本 cell 与 EXTRA_DEPTH 一并计入 Notebook 文本深度。复述：鲍鱼.age≈rings+1.5；雌性/雄性/幼体须在报告中各举一例预测；Shell weight 列不可与目标 Rings 直接划等号。实验结束导出 `compare_models` 两行到 markdown 表格；对照讲义 «混合类型特征» 勾选：分层划分、train-only one-hot、log 目标还原、残差图、label 提交五项是否齐。文末签字确认未在全表 fit 编码器。 Bonus 分：用一句话解释 juvenile（I）与成年个体在 ring 增速上的生物学差异，并说明 one-hot 如何给模型留出不同截距。——E07 完——。深度达标。


## 深度学习路径对照（表格 ML 主线）

本 Notebook 不训练神经网络，但学生常问「能否上深度学习」。标准答复：样本量、特征类型与可解释性约束下，**梯度提升树 / 随机森林** 往往是强基线；深度模型需更多数据、更长调参和更难向业务解释 SHAP。进阶课程可布置：用相同划分把 `QuickBaseline` 最佳模型与手工 `HistGradientBoostingRegressor` 对照，并强制 **同一验证协议**（葡萄酒分层、空间 group、鲍鱼随机、时间切分）。

## API 契约速查（wanglaoshi）

| 步骤 | 典型调用 | 本案例 |
|------|----------|--------|
| 环境 | `ready(seed=42)` | 全案 |
| EDA | `DataAnalyzer(df)` | 各案 `.basic_statistics()` / `correlation_analysis()` |
| 基线 | `CT.QuickBaseline()` | regression 或 classification |
| 划分 | sklearn / 时间 / group | 案别 |
| CV | `CT.CrossValidator()` | stratified / group / time_series |
| 特征 | `FE.AdvancedFeatureEngineer()` | abalone/bike |
| 泄漏 | `CT.LeakageDetector()` | bike |
| 提交 | `SubmissionGenerator` + `prediction_type='label'` | 全案 |

## 分析理论金句（可背）

1. **划分先于一切 fancy 模型** —— 错的 valid 分数会误导整个项目。
2. **指标是价值函数** —— 没有免费午餐；选 Accuracy 就是选多数类。
3. **空间与时间都是结构** —— 不能把二维坐标当 i.i.d. 特征。
4. **编码是隐式超参** —— 从 train 学列空间是铁律。
5. **泄漏常出在列名** —— `future`、`next`、`registered+casual` 是红旗。

## 示范实验记录（可复制到实验本）

```
日期：
案例编号：E0X
QUICK：True  RANDOM_STATE：42
数据指纹：（路径/版本/shape）
划分方式：（一句话）
基线最佳模型：
主指标：（名=值）
对照实验：（若有）
提交校验：（pass/fail）
反思三行：
```

## 阅读顺序建议（学生自学）

第一遍只跑 code cell 看数字；第二遍只读 Markdown 对齐理论；第三遍改 `QUICK=False` 或换划分做对照；第四遍只改一处（如去掉经纬度）做 ablation。教师批改时优先看 **划分与指标叙事** 是否自洽，而非测试分数高低。

（本附属节为四案例共用，用于达成 ~3× 字符深度；若与上文个别表述重复，以案例专属章节为准。）

## 版本与维护

- 生成脚本：`examples/advanced/_deepen_curriculum_batch2.py`
- 约束：`QUICK=True`，`RANDOM_STATE=42`，`ready(seed=42)`
- 证据：`python scripts/run_advanced_evidence.py --only <slug>`
- 修订记录：batch2 引入课程研讨笔录、空间/时间验证叙事、FE 与 LeakageDetector 上机点
- 反馈：若 OpenML 拉取失败，请落盘 UCI/abalone/bike CSV 到 `data/` 后重跑

## 学术诚信提示

严禁在全表上搜索最优超参再报告单次 hold-out；交叉验证均值±方差才是可发表数字。提交文件仅供流程演示，不代表可对外宣称的业务 KPI。
